In [3]:
import time
import pandas as pd
import great_expectations as ge
from great_expectations.execution_engine import PandasExecutionEngine
from great_expectations.validator.validator import Validator
from great_expectations.core.expectation_suite import ExpectationSuite

In [5]:
def run_stage1_validation(data_path):
    print(f"\n==========================================")
    print(f"STAGE 1 VALIDATION (Great Expectations) ON: {data_path}")
    print(f"==========================================")
    
    # 1. Load dataset & filter relevant transaction types
    df_raw = pd.read_csv(data_path)
    df_filtered = df_raw[df_raw['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
    
    # Measure validation timing (T_valid)
    t_start_valid = time.time()
    
    # 2. Setup Great Expectations Validator for modern GE versions
    suite = ExpectationSuite(name="paysim_ingestion_suite")
    engine = PandasExecutionEngine()
    validator = Validator(
        execution_engine=engine,
        expectation_suite=suite,
        in_memory_dataset=df_filtered
    )
    
    # 3. Define Declarative Expectations (Contracts)
    target_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
    
    # Rule 1: Columns must not contain null/missing values
    for col in target_cols:
        validator.expect_column_values_to_not_be_null(column=col)
        
    # Rule 2: Transaction amounts must be non-negative
    validator.expect_column_values_to_be_between(column='amount', min_value=0)
    
    # Rule 3: Balance columns must exist in schema
    validator.expect_column_to_exist(column='oldbalanceOrg')
    validator.expect_column_to_exist(column='newbalanceOrig')
    
    # 4. Validate Micro-batch Data
    validation_results = validator.validate()
    t_end_valid = time.time()
    
    # 5. Extract Metrics
    t_valid_duration = t_end_valid - t_start_valid
    success = validation_results["success"]
    stats = validation_results["statistics"]
    
    print(f"Validation Status: {'SUCCESS (Pristine Data)' if success else 'FAILED (Anomalies Detected)'}")
    print(f"Validation Time (T_valid): {t_valid_duration:.4f} seconds")
    print(f"Evaluated Rules: {stats['evaluated_expectations']}")
    print(f"Passed Rules: {stats['successful_expectations']}")
    print(f"Failed Rules: {stats['unsuccessful_expectations']}")
    
    return validator, validation_results, t_valid_duration

# Execute validation on pristine data
if __name__ == "__main__":
    validator, results, t_valid = run_stage1_validation('Dataset.csv')


STAGE 1 VALIDATION (Great Expectations) ON: Dataset.csv


TypeError: active_batch cannot be None

In [6]:
import time
import pandas as pd
import great_expectations as ge

def run_stage1_validation(data_path):
    print(f"\n==========================================")
    print(f"STAGE 1 VALIDATION (Great Expectations) ON: {data_path}")
    print(f"==========================================")
    
    # 1. Load dataset & filter relevant transaction types
    df_raw = pd.read_csv(data_path)
    df_filtered = df_raw[df_raw['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
    
    # Measure validation timing (T_valid)
    t_start_valid = time.time()
    
    # 2. Build Ephemeral Context & Batch
    context = ge.get_context(mode="ephemeral")
    
    # Create an in-memory data asset
    datasource = context.sources.add_pandas("in_memory_datasource")
    data_asset = datasource.add_dataframe_asset(name="paysim_asset")
    batch_request = data_asset.build_batch_request(dataframe=df_filtered)
    
    # 3. Create Expectation Suite
    suite = context.add_or_update_expectation_suite(expectation_suite_name="paysim_ingestion_suite")
    validator = context.get_validator(batch_request=batch_request, expectation_suite=suite)
    
    # 4. Define Declarative Rules
    target_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
    
    # Rule 1: Columns must not contain null/missing values
    for col in target_cols:
        validator.expect_column_values_to_not_be_null(column=col)
        
    # Rule 2: Transaction amounts must be non-negative
    validator.expect_column_values_to_be_between(column='amount', min_value=0)
    
    # 5. Validate Micro-batch Data
    validation_results = validator.validate()
    t_end_valid = time.time()
    
    # 6. Compute & Display Metrics
    t_valid_duration = t_end_valid - t_start_valid
    success = validation_results["success"]
    stats = validation_results["statistics"]
    
    print(f"Validation Status: {'SUCCESS (Pristine Data)' if success else 'FAILED (Anomalies Detected)'}")
    print(f"Validation Time (T_valid): {t_valid_duration:.4f} seconds")
    print(f"Evaluated Rules: {stats['evaluated_expectations']}")
    print(f"Passed Rules: {stats['successful_expectations']}")
    print(f"Failed Rules: {stats['unsuccessful_expectations']}")
    
    return validator, validation_results, t_valid_duration

# Execute validation on pristine data
if __name__ == "__main__":
    validator, results, t_valid = run_stage1_validation('Dataset.csv')


STAGE 1 VALIDATION (Great Expectations) ON: Dataset.csv


AttributeError: 'EphemeralDataContext' object has no attribute 'sources'